<a href="https://colab.research.google.com/github/padmasundarg95/Project-NEXUS-Agent/blob/main/Project_NEXUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#--------------------------------------------1. SCRAPER CODE-------------------------------------------#
!pip install python-docx #Install python docx for word document
import requests #requests is to get requests after hitting a server
import bs4
from bs4 import BeautifulSoup #used to parse the recieved text in a readable format
from google.colab import files #required for downloading the word document
import time #used for giving sleep time
from docx import Document #used for importing Document for word document
url = 'https://padmasundarg.blogspot.com/' #defining url
data = [] #defining a bucket to collect data
blogs_content = [] #defining a bucket to collect all the blogs content

#FUNCTION 1 FOR GETTING THE INDIVIDUAL BLOG LINKS
def content(blog_link):
      try: #try & except to act as a safety net
        res = requests.get(blog_link) #passing the blog_link to get the response
        soup1 = BeautifulSoup(res.text,'html.parser') #using beautifulsoup to get the parsed output
        body = soup1.find('div',class_='post-body-container') #giving CSS Selector to get the blog content
        if body: #conditional formatting
          clean_body = body.get_text().split() #splits the sentences into word by word
          cleaned_data = " ".join(clean_body) #the cleaned words are joined and saved in cleaned_data
          data.append(cleaned_data) #appending all blogs data to data = [] bucket
        return cleaned_data #this function returns cleaned data
      except Exception as e:
        print("Link not found") #safety mechanism to handle in case the blog link is not found

#FUNCTION 2 FOR GETTING THE CLEANED DATA IN SLICE FORMS/CHUNKS FORM - WILL BE USED IN RAG
def chunks(cleaned_data):
  words = cleaned_data.split() #cleaned data is split into words
  chunk_size = 30 #assigning chunk_size as 30
  overlap = 10 #assigning the overlap required between two words as 10 - semantic handshake to avoid getting the context lost in split
  step = chunk_size - overlap #defining step
  slices_basket = [] #defining an empty bucket to collect all the chunks/slices, defining this inside the function as the func needs to return this
#DATA SHREDDER
  for i in range(0,len(words),step): #measuring tape with intervals
    a = words[i:i+chunk_size] #slicing option with overlap to ensure AI doesn't lose the context
    clean_slice = " ".join(a) #using a space, stringing the words back together
    slices_basket.append(clean_slice) #putting all the slices to the empty bucket
  return slices_basket #this function returns all the slices

#-----------------------------------MAIN LOOP WHERE THE ABOVE DEFINED TWO FUNCTIONS ARE CALLED-----------------------------------#
while url: #run till the url becomes none
    response = requests.get(url) #get the response from the main url defined in the start of the code
    soup = BeautifulSoup(response.text,'html.parser') #parse the received response
    blog_title = soup.find_all('h3',class_='post-title') #using the CSS Selector for the title, finding all titles and saving it in blog_tite
    for i in blog_title: #loop to go to each and every blog title
      clean_title = i.find('a').get_text() #from the title's CSS Selector, just take the title's text version removing all html tags
      blog_link = i.find('a')['href'] #from the title's CSS Selector, fetch the link residing in a.href
      blog_text = content(blog_link) #calling function 1
      blog_slices = chunks(blog_text) #calling function 2
      titles_links_body = {"Blog title":clean_title,"Blog link":blog_link,"Content":blog_text,"chunks":blog_slices} #based on the above fetched details
      blogs_content.append(titles_links_body) #append all the data to blogs_content [] bucket
    nextpage = soup.find('a',class_='blog-pager-older-link') #using CSS Selector for the next page, get the nextpage link details
    if nextpage: #conditional logic
      url = nextpage['href'] #fetching the nextpage url to go to the next page
    else:
      url = None #if url for next page is not found, url becomes none and the while loop stops
print(f"Scraped all {len(blogs_content)} blogs") #check if displays the exact total count of blogs

Scraped all 63 blogs


In [6]:
#-------------------------------------------------2. AI LOGIC------------------------------------------------#
!pip install -q -U google-genai
from google import genai
from google.colab import files
from docx import Document
from google.colab import userdata
import pandas as pd
import re
from docx.shared import RGBColor, Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches
api_key = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Persona Prompting
narrative_rules = """You are a 'Theme Curator Agent'.
Analyze the provided blog and provide a structured response using ONLY these labels:

GENRE: [Classification]
CREATIVE STRENGTH: [Key technique]
TRANSFERABLE SKILL: [Business competency]
SUMMARY: [Give a sentiment analysis summary and describe the analysis in five catchy sentences]
CRITIC RATING: [X.X]/5
STRENGTHS: [List 2 points, do not use any brackets, just use comma separation]
IMPROVEMENTS: [List 2 points, do not use any brackets, just use comma separation]
GRAMMAR CHECK: [Note any errors, do not use any brackets, just use comma separation]
"""
#Defining the models and linking it the prompts to get the blog analysis done
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY')) #configuring the api_key in genai with the secret key
def get_ai_analysis(story, title): #function for the AI part
    retries = 0
    while retries < 5:
        try:
            response = client.models.generate_content(
                model='gemma-3-27b-it',
                contents=f"{narrative_rules}\nTITLE: {title}\nSTORY: {story}"
            ) #using the model, generating the analysis by using the pre defined rules
            return response.text #function returns the analysis data
        except Exception as e: #handling exception in case if the model hit its quota
            if "429" in str(e):
                print(f"Quota hit. Retrying in 60s...")
                time.sleep(60) #wait period of 60 seconds
                retries += 1 #next cycle begins with retries = retries + 1
            else:
                print(f"ERROR: {e}")
                return None
    return None

In [7]:
#-------------------------------------------3. SANITIZING CODE----------------------------------------------#
def document_formatting(doc, title, review_text):
    heading = doc.add_heading(title, level=1) #making the title of the blog as the second main heading

    for run in heading.runs: #for loop to go to every
        run.font.name, run.font.size, run.font.color.rgb = 'Arial', Pt(16), RGBColor(31, 78, 121) #Add Title Heading (Blue, Arial, Size 16)

    for line in review_text.split('\n'): #line will go to every single line/row item that review_text.split gives, \n is for newline
        clean = line.replace('**', '').replace('*', '').replace('---', '').replace('#','').replace('##','').strip() #Removing all markdown symbols like *, ---, # [Santizing the payload]

        if not clean or len(clean) < 3:
          continue #conditional logic to remove data noise i.e empty spaces and words of length < 3 [Noise Filtering]

        p = doc.add_paragraph() #This is similar to enter in word document
        p.paragraph_format.space_after = Pt(6) #paragraph_format - settings menu for a specific paragraph in word, space_after=pt(6) - to tell the word to leave space(6 points) before it starts the next line

#-----------------------------FORMATTING FOR RATING --------------------------------#
        if "RATING" in clean.upper(): #conditional logic to check if rating is there in the cleaned response(upper is given to avoid any case wise characters mismatch/Normalizing for case sensitivity)
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER #align the rating to the center [Paragraph-Level Metadata]
            run = p.add_run(f"{clean}⭐") #add a star emoji to the cleaned rating output
            run.bold, run.font.size, run.font.color.rgb = True, Pt(12), RGBColor(192, 0, 0) #formatting the rating line in bold, centered and red
#-----------------------------LABELS LOGIC--------------------------------------------#
        elif ":" in clean: #conditional logic to identify key value / label content pairs
            label, val = clean.split(":", 1) #splitting the cleaned data based on : presence and splitting it just once and the left side of : is label and right side is val
            l_run = p.add_run(f"{label.strip().upper()}: ") #removing the unnecessary spaces from label and making it upper case
            l_run.bold = True #making the label bold
            l_run.font.color.rgb = RGBColor(31, 78, 121) #making the label Blue color
            p.add_run(val.strip()) #pastes the cleaned value data into paragraph
#-----------------------------CONDITIONAL LOGIC FOR THE GENERIC SENTENCES====================#
        else:
            p.style = 'List Bullet' #list the generic sentences in List Bullet form
            p.add_run(clean) #add the entire cleaned text to the document paragraph as it is

    doc.add_page_break() #Moving to the next page

In [8]:
#----------------------------------4. DOCUMENT COVER PAGE CODE-------------------------------------#

doc = Document() #Opening a new word document
#Default Format for the Document - SETTING THE GLOBAL[DEFAULT] FORMAT TO NORMAL, ARIAL, SIZE 11
style = doc.styles['Normal']
style.font.name = 'Arial'
style.font.size = Pt(11)
#--------------------------COVER PAGE - VERTICAL & HORIZONTAL CENTERING--------------------#
for _ in range(12): # Adding empty paragraphs to push title to vertical center [White space management]
    doc.add_paragraph() #similar to enter in word doc [technically creates a new object container that can have its own space and alignment settings]

#--------------------------COVER PAGE - MAIN TITLE FORMAT----------------------------------#
t_p = doc.add_paragraph() #similar to enter in word doc
t_p.alignment = WD_ALIGN_PARAGRAPH.CENTER #aligning the title to center
t_run = t_p.add_run('Project NEXUS') #giving the title name
t_run.bold = True #setting it to bold
t_run.font.size = Pt(28) #setting it to size 28
t_run.font.name = 'Arial' #setting it to Arial font

#--------------------------COVER PAGE - SUB TITLE FORMAT----------------------------------#
s_p = doc.add_paragraph() #similar to enter in word doc
s_p.alignment = WD_ALIGN_PARAGRAPH.CENTER #aligning the sub title to center
s_p.space_before = Pt(12) #giving a space of 12 pointers before the start of the sub title
s_run = s_p.add_run('Advanced Agentic Framework for Multi-Modal Narrative Intelligence')
s_run.bold = True #setting it to bold
s_run.italic = True #setting it to italic
s_run.font.size = Pt(14) #setting it to size 14
s_run.font.name = 'Arial' #setting it to Arial font
s_run.font.color.rgb = RGBColor(80, 80, 80)

# Spacing
doc.add_paragraph()

#-------------------------META DATA/OTHER INFORMATION SECTION - FORMATTING-------------------#
meta_p = doc.add_paragraph() #similar to enter in word
meta_p.alignment = WD_ALIGN_PARAGRAPH.LEFT #Left alignment
meta_p.space_before = Pt(48) # Push it down the page for a clean look

def other_informations(label, value): #function to add the data in different lines for a cleaner look
    p = doc.add_paragraph() #similar to enter in word
    p.paragraph_format.left_indent = Inches(1.0) # [Negative space design] paragraph.format - settings for that specific block of text, left indent of one inch to push away from margin for a better look
    run_label = p.add_run(f"{label}: ") #add the label to the paragraph
    run_label.bold = True #setting label to bold
    run_label.font.name = 'Arial' #setting it to Arial font
    run_label.font.size = Pt(10) #setting it to size 10
    run_val = p.add_run(value) #add the value to the paragraph
    run_val.font.name = 'Arial' #setting value font as Arial
    run_val.font.size = Pt(10) #setting value size as 10

#CALLING THE FUNCTION - OTHER_INFORMATIONS
other_informations("Prepared by", "PadmaSundar")
other_informations("Role", "Senior Business Analyst - Dell")
other_informations("Pipeline Version", "NEXUS v1.1 (Agentic)")
other_informations("Date", "April 2026")

doc.add_page_break() #similar to enter in word

In [10]:
#-------------------------------------------5. LOOP TO RUN AI---------------------------------------#
for i, blog in enumerate(blogs_content[:]): #This loop runs the AI once and then writes the result immediately #setting it to 1 blog for testing purpose [Sequential processing]
    title = blog.get('Blog title', 'Untitled') #to cover both Blog title and the ones that are not titled (fallback mechanism/safety net)
    content = blog.get('Content', '') #get the content (if not take an empty value to prevent it from crashing)
    print(f"Agent is analyzing: {title}")
    raw_review = get_ai_analysis(content, title) #calling the function defined earlier by passing content and title arguemnts fetched from blogs_content
    if raw_review: #conditional logic - if there is a data returned, then call the formatting logic to apply all the formats
        document_formatting(doc, title, raw_review)
        print(f"Formatted and added to Document.")
        time.sleep(15) #Sleep of 15 seconds/Safety gap for the API to prevent overload [Cool down logic/Rate parsing]

doc.save('Project_NEXUS_v1.docx') #save the document
files.download('Project_NEXUS_v1.docx') #download the document

Agent is analyzing: Beyond the Veil🌟
Formatted and added to Document.
Agent is analyzing: Sudha and her Eclairs Chocolate🍬
ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Agent is analyzing: Moonstruck with this one forever🌙
ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Agent is analyzing: Chapter 5: Mumbai Local Trains: Where Different Stories meet
Formatted and added to Document.
Agent is analyzing: Chapter 4: Movie Stars and My Childhood Fantasies – Mumbai Diaries🛺
Formatted and added to Document.
Agent is analyzing: oru light kattan☕
Formatted and added to Document.
Agent is analyzing: Mumbai Diaries {Chapter 3: The Lunchbox Window}
Formatted and added to Document.
Agent is ana

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#----------------------------------------6. EMAIL TRIGGER---------------------------------------------------#
import smtplib #manages the handshake between python script and gmail server
from email.mime.multipart import MIMEMultipart #container to hold different types of data into a single package [Encapsulation]
from email.mime.text import MIMEText #manages the body of the email,translates the message into a format that an email box can read as a standard message
from email.mime.base import MIMEBase #base for anything that isn't a plain text, provides the structure to hold the document
from email import encoders #data translator for the document,protects the data so that it can pass through pipes of internet without getting corrupted
import os
from google.colab import userdata

def nexus_agent_mail_delivery(file_name, recipient_email):

    # 1. AGENT IDENTITY (Sender's Info & password)
    sender_email = "padmasundarg95@gmail.com"
    app_password = userdata.get('EMAIL_PASSWORD')# 16 character code from Google's app passwords [secret key]

    # 2. CREATE THE 'PACKAGE' (The Envelope)
    message = MIMEMultipart() #multipart container that has different sections for different types of data - text, word document etc
    message['From'] = sender_email #sender's email (already defined above)
    message['To'] = recipient_email #recipient email (will be passed as an arguemnt for the function while calling it)
    message['Subject'] = "Project NEXUS: E2E Agentic Workflow - Autonomous Narrative Synthesis & Multi-Channel Dispatch"

    # The 'Letter' inside the envelope
    body = f"""
    Hi PSG,

The NEXUS Agent has successfully executed the end-to-end (E2E) processing cycle for your content portfolio.

MISSION SUMMARY:
- Ingestion: 60+ Blogs Analyzed
- Processing Engine: Gemini-Powered Semantic Reasoning
- Output: Integrated Narrative Intelligence Report (Attached)

The agent has completed the thematic synthesis across your portfolio.
This audit is now formatted and ready for your review or RAG-tier ingestion.

System Status: PHASE 1 DEPLOYMENT SUCCESSFUL

Regards,
NEXUS Intelligence Agent
    """
    message.attach(MIMEText(body, 'plain')) #mimetext converts python string into mime object - specialized wrapper, and plain tells the email system to read it as text

    # 3. ATTACH THE WORD DOCUMENT TO THE EMAIL
    try:
        with open(file_name, "rb") as attachment: #opens the file in its rawest form, rb - read binary (1s and 0s)
            part = MIMEBase("application", "octet-stream") #telling the email server that we are sending a binary file
            part.set_payload(attachment.read()) #puts the binary data from word doc into the container

        encoders.encode_base64(part) #scrambles the binary data into base64/safe text format so that it can travel safely via internet
        part.add_header("Content-Disposition", f"attachment; filename= {file_name}") #shipping labale - this tells the recipient's email app that this is an attachment/file to be downloaded with a specific name
        message.attach(part) #attaches the file into the package

        # 4. THE DELIVERY (The Handshake)
        server = smtplib.SMTP('smtp.gmail.com', 587) #connecting to gmail's post office (587 - loading dock for secure modern emails)
        server.starttls() # This encrypts the connection (the armored truck), locks the truck so our password and file cannot be stolen during the transit
        server.login(sender_email, app_password) #security checkpoint - show the sender's email and password
        server.send_message(message) #package is officially handed over to google for delivery
        server.quit() #closes the connection to the server properly, like hanging up a phone call instead of cutting the line

        print(f"✅ NEXUS Agent: Report successfully delivered to {recipient_email}!")

    except Exception as e:
        print(f"❌ Failed to Deliver: {e}")

nexus_agent_mail_delivery("Project_NEXUS_v1.docx", "psg771111@gmail.com") #triggering the agent by calling the function


✅ NEXUS Agent: Report successfully delivered to psg771111@gmail.com!
